# Quét ảnh ra vật thể + từ tiếng Anh — BẢN FLORENCE-2

## Vì sao thử Florence-2 thay cho chuỗi RAM++ → Grounding DINO?

Pipeline v7/v8 là **chuỗi 4–5 model nối tiếp** (RAM++ → lọc từ điển → Grounding DINO/OWLv2 → SAM → CLIP).
Mỗi khớp nối đều rò rỉ lỗi theo cách riêng:
- RAM++ bỏ sót tag → vật thể **không bao giờ** được tìm (RAM++ là cận trên recall của cả hệ thống).
- Map nhãn thô của Grounding DINO về tag gốc dễ sai → false positive có hệ thống.
- Ngưỡng detector lệch giữa các batch prompt → phải chế thêm calibrate/dynamic threshold.

**Florence-2** (Microsoft) là 1 model duy nhất làm cả 2 việc: nhìn ảnh → tự đề xuất box **kèm nhãn mở**
(không cần cho trước danh sách tag), qua 2 task:
- `<OD>` — object detection với nhãn ngắn gọn ("car", "dog", "traffic light"...).
- `<DENSE_REGION_CAPTION>` — mô tả từng vùng bằng cụm từ giàu hơn ("a red vintage car"...) → bắt được vật
  mà `<OD>` gọi tên chung chung hoặc bỏ qua; ta trích **danh từ chính** của cụm làm nhãn flashcard.
- (tuỳ chọn) `<CAPTION_TO_PHRASE_GROUNDING>` — "tiêm" từ bạn muốn tìm, thay cho `extra_tags` cũ.

SAM vẫn giữ nguyên vai trò khâu cuối: xác thực box (mask quá nhỏ → loại) + cắt nền trong suốt cho flashcard.
Bộ đánh giá dùng bản ĐÃ SỬA của v8: so nhãn bằng alias + hyponym (man↔person), micro-average, giao thức
federated cho Open Images, và tuỳ chọn giao từ vựng khi đo trên COCO.

> **Lưu ý:** Florence-2 KHÔNG trả về điểm tin cậy cho từng box (model sinh chuỗi, không phải detector
> truyền thống). Ta gán pseudo-score theo nguồn (`<OD>` = 0.90, dense caption = 0.75) chỉ để sắp thứ tự
> trong NMS; nếu cần điểm thật để lọc, bật `ENABLE_CLIP_SCORING` — CLIP chấm điểm crop↔nhãn.

Chạy trên Google Colab, GPU T4 trở lên (Florence-2-large ~1.5GB fp16 + SAM vit_h ~2.4GB).


## Bước 1 — Cài đặt thư viện

Sau khi cell chạy xong hãy **Restart session** rồi chạy tiếp từ Bước 2.

In [ ]:
# Thư viện cơ bản
!pip install -q torch torchvision pillow numpy opencv-python matplotlib

# Florence-2 cần transformers + timm + einops (code model tải qua trust_remote_code)
!pip install -q "transformers==4.49.0" timm einops

# SAM (Segment Anything) — giữ nguyên vai trò xác thực + cắt nền như bản v7/v8
!pip install -q segment-anything

# NLTK: POS-tagging để trích danh từ chính từ caption + WordNet cho so khớp nhãn khi đánh giá
!pip install -q nltk

# FiftyOne — chỉ cần cho bộ test Internet-50 (Open Images V7) ở Bước 8
!pip install -q fiftyone

print("Cài đặt xong. VUI LÒNG RESTART SESSION TRƯỚC KHI CHẠY BƯỚC 2!")


## Bước 2 — Load Florence-2 + SAM

- `microsoft/Florence-2-large` (~0.77B tham số). Đổi sang `microsoft/Florence-2-base` nếu thiếu VRAM/muốn nhanh.
- Code gốc của Florence-2 khai báo import `flash_attn` (không cài được trên Colab T4) — patch bên dưới gỡ import
  đó ra và dùng attention SDPA mặc định của PyTorch, đây là workaround chuẩn được cộng đồng dùng rộng rãi.


In [ ]:
import os
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Đang chạy trên: {device} ({torch_dtype})")

FLORENCE_MODEL_ID = "microsoft/Florence-2-large"   # đổi "microsoft/Florence-2-base" nếu thiếu VRAM
SAM_MODEL_TYPE = "vit_h"                            # đổi "vit_b" nếu thiếu VRAM

# ---------- Florence-2 (patch bỏ import flash_attn — không cài được trên Colab T4) ----------
from unittest.mock import patch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports


def _fixed_get_imports(filename):
    imports = get_imports(filename)
    if not str(filename).endswith("modeling_florence2.py"):
        return imports
    return [i for i in imports if i != "flash_attn"]


with patch("transformers.dynamic_module_utils.get_imports", _fixed_get_imports):
    florence_model = AutoModelForCausalLM.from_pretrained(
        FLORENCE_MODEL_ID, torch_dtype=torch_dtype, trust_remote_code=True,
        attn_implementation="sdpa",
    ).to(device).eval()
    florence_processor = AutoProcessor.from_pretrained(FLORENCE_MODEL_ID, trust_remote_code=True)
print(f"✅ Đã load Florence-2 ({FLORENCE_MODEL_ID}).")

# ---------- SAM (giữ nguyên checkpoint như bản v7/v8) ----------
SAM_CHECKPOINT_URLS = {
    "vit_b": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
    "vit_l": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth",
    "vit_h": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
}
SAM_CKPT = f"/content/checkpoints/{SAM_CHECKPOINT_URLS[SAM_MODEL_TYPE].split('/')[-1]}"
os.makedirs("/content/checkpoints", exist_ok=True)
if not os.path.exists(SAM_CKPT):
    !wget -q -nc -O {SAM_CKPT} {SAM_CHECKPOINT_URLS[SAM_MODEL_TYPE]}

from segment_anything import sam_model_registry, SamPredictor
sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CKPT).to(device)
sam_predictor = SamPredictor(sam)
print(f"✅ Đã load SAM ({SAM_MODEL_TYPE}).")

# ---------- CLIP (tuỳ chọn — chấm điểm thật cho box vì Florence-2 không có score) ----------
ENABLE_CLIP_SCORING = False   # bật True nếu muốn lọc/re-rank bằng điểm CLIP crop↔nhãn
if ENABLE_CLIP_SCORING:
    from transformers import CLIPModel, CLIPProcessor
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    print("✅ Đã load CLIP.")

# ---------- NLTK ----------
import nltk
for pkg in ["wordnet", "omw-1.4", "words", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Bỏ qua gói {pkg}: {e}")
print("✅ Đã tải dữ liệu NLTK.")


## Bước 3 — Chuẩn hoá nhãn + so khớp nhãn (kế thừa bản v8 đã sửa)

- `caption_to_label`: trích **cụm danh từ cuối** của caption ("a red vintage car parked..." → "car",
  "traffic light on pole" → "traffic light") — đây là từ đưa lên flashcard.
- `is_scene_background_word`: chỉ áp cho tag MỘT TỪ (đã fix bug "traffic light" của bản cũ).
- `labels_match`: đồng nghĩa WordNet + bảng alias + quan hệ hyponym→hypernym CÓ HƯỚNG (man → person,
  sedan → car) — dùng cho đánh giá, để không phạt oan nhãn cụ thể hơn ground truth.


In [ ]:
import re
import nltk
from nltk.corpus import wordnet, words as nltk_words
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
ENGLISH_WORDS = set(w.lower() for w in nltk_words.words())

STRICT_DICTIONARY_FILTER = True   # True: chỉ nhận nhãn có trong từ điển (COCO); False: bộ lọc mềm (Internet)

SCENE_BACKGROUND_STOPWORDS = {
    "wall", "floor", "ceiling", "sky", "room", "background", "shadow", "light",
    "photo", "image", "picture", "texture", "pattern", "surface", "reflection",
    "scene", "view", "outdoor", "indoor", "ground", "area", "space",
    # SỬA: tên CĂN PHÒNG là bối cảnh, không phải vật cắt được thành flashcard
    # ("kitchen" từng lọt ra thành thẻ ở ảnh bếp F2-v12)
    "kitchen", "bathroom", "bedroom", "hallway", "interior",
}

# Giới từ dùng để cắt caption: "a red car parked ON the street" -> chỉ giữ "a red car parked"
_PREPOSITIONS = {
    "in", "on", "at", "with", "of", "near", "by", "under", "over", "behind", "beside",
    "above", "below", "around", "against", "across", "during", "from", "to", "next", "front",
}


def normalize_word(word):
    word = word.lower().strip()
    word = lemmatizer.lemmatize(word, pos="n")
    return word


def is_valid_dictionary_word(word):
    word = word.lower().strip()
    if not word.replace("-", "").isalpha():
        return False
    if word in ENGLISH_WORDS or wordnet.synsets(word):
        return True
    lemma = normalize_word(word)
    return lemma in ENGLISH_WORDS or bool(wordnet.synsets(lemma))


def is_scene_background_word(tag):
    tokens = tag.strip().lower().split()
    if len(tokens) != 1:   # fix bug bản cũ: KHÔNG xét head của tag nhiều từ ("traffic light" từng bị loại oan)
        return False
    return normalize_word(tokens[0]) in SCENE_BACKGROUND_STOPWORDS


def caption_to_label(caption):
    """Trích cụm danh từ CUỐI CÙNG trước giới từ đầu tiên của caption làm nhãn flashcard.
    "a red vintage car parked on the street" -> "car"; "traffic light" -> "traffic light"."""
    toks = re.findall(r"[a-z]+", caption.lower())
    cut = []
    _CONJUNCTIONS = {"and", "or", "an", "a", "the"}   # SỬA: pos_tag đôi khi gán nhầm "and" là danh từ
    for t in toks:                                     # -> nhãn rác kiểu "and socket"
        if t in _PREPOSITIONS:
            break
        if t in _CONJUNCTIONS:
            continue
        cut.append(t)
    if not cut:
        return None
    # SỬA: pos_tag hay gán nhầm danh từ trong cụm ngắn không ngữ cảnh thành động từ
    # ("microwave oven" -> VB VBN -> mất trắng cả nhãn). Một token được coi là danh-từ-ứng-viên
    # nếu POS là NN* HOẶC từ đó có nghĩa danh từ trong WordNet — trừ hư từ (a/the/and...).
    _FUNC_WORDS = {"a", "an", "the", "this", "that", "these", "those", "and", "or",
                   "it", "its", "his", "her", "their", "my", "your", "our", "some", "any"}
    tagged = nltk.pos_tag(cut)
    run, best = [], []
    for w, p in tagged:
        is_nounish = (w not in _FUNC_WORDS and len(w) > 1
                      and (p.startswith("NN") or bool(wordnet.synsets(w, pos=wordnet.NOUN))))
        if is_nounish:
            run.append(normalize_word(w))
        else:
            if run:
                best = run
            run = []
    if run:
        best = run
    if not best:
        return None
    # Chỉ giữ cụm 2 danh từ nếu nó là từ ghép có thật trong WordNet ("traffic light", "cell phone");
    # nếu không, lấy danh từ cuối cùng — tránh nhãn kiểu "vintage car" thay vì "car".
    if len(best) >= 2 and wordnet.synsets("_".join(best[-2:])):
        return " ".join(best[-2:])
    return best[-1] or None


# ====== Bộ lọc "danh từ chỉ VẬT THỂ CỤ THỂ" (đồng bộ với bản v8.4/8.5) ======
# Caption của Florence-2 đôi khi ra danh từ trừu tượng ("fun", "moment", "celebration") — lọc bằng
# WordNet: nghĩa phổ biến của từ phải là vật chụp được (lexname/chuỗi hypernym về physical_entity).
ENABLE_CONCRETE_FILTER = True

_PHYSICAL_ROOTS = {wordnet.synset("physical_entity.n.01")}
_CONCRETE_LEXNAMES = {"noun.artifact", "noun.animal", "noun.food", "noun.person",
                      "noun.plant", "noun.object", "noun.body", "noun.substance"}
_CONCRETE_WHITELIST = {
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog",
    "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite",
    "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle",
    "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant",
    "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
    "teddy bear", "hair drier", "toothbrush", "ring", "sign", "kite",
}


def _sense_is_concrete(syn, max_up=20):
    if syn.lexname() in _CONCRETE_LEXNAMES:
        return True
    frontier = {syn}
    for _ in range(max_up):
        if frontier & _PHYSICAL_ROOTS:
            return True
        nxt = set()
        for s in frontier:
            nxt.update(s.hypernyms())
        if not nxt:
            return False
        frontier = nxt
    return False


def is_concrete_object_word(tag):
    t = tag.strip().lower()
    if t in _CONCRETE_WHITELIST:
        return True
    key = t.replace(" ", "_")
    syns = wordnet.synsets(key, pos=wordnet.NOUN)
    if not syns and " " in t:
        syns = wordnet.synsets(t.split()[-1], pos=wordnet.NOUN)
    if not syns:
        return False if wordnet.synsets(key) else None
    if _sense_is_concrete(syns[0]):
        return True
    return sum(_sense_is_concrete(s) for s in syns[1:3]) >= 2


def check_synonyms(word1, word2):
    w1, w2 = word1.lower().strip(), word2.lower().strip()
    if w1 == w2:
        return True
    syns1 = set()
    for syn in wordnet.synsets(w1.replace(" ", "_")):
        for lemma in syn.lemmas():
            syns1.add(lemma.name().lower().replace("_", " "))
    return w2 in syns1


LABEL_ALIASES = {
    "person": {"man", "woman", "boy", "girl", "child", "kid", "lady", "guy", "people",
               "player", "pedestrian", "rider", "baby", "human"},
    "tv": {"television", "monitor", "screen", "tv set"},
    "couch": {"sofa", "settee"},
    "motorcycle": {"motorbike"},
    "bicycle": {"bike", "cycle"},
    "cell phone": {"phone", "mobile phone", "smartphone", "telephone"},
    "sports ball": {"ball", "soccer ball", "basketball", "baseball", "tennis ball", "football", "volleyball"},
    "potted plant": {"plant", "houseplant", "flowerpot", "pot plant"},
    "dining table": {"table", "desk"},
    "hair drier": {"hair dryer", "dryer", "blow dryer"},
    "airplane": {"plane", "aircraft", "jet"},
    "boat": {"ship", "sailboat", "canoe", "kayak"},
    "car": {"automobile", "suv", "sedan", "taxi", "van"},
    "remote": {"remote control", "controller"},
    "handbag": {"purse", "bag"},
    "backpack": {"rucksack", "knapsack"},
    "wine glass": {"wineglass", "glass"},
    "traffic light": {"stoplight", "traffic signal"},
}

# SỬA — từ phân tích 96 lượt từ sót trên Internet-50: các lớp phạm trù/bộ phận của Open Images
# mà Florence gọi bằng tên khác. Thiếu alias thì detection đúng vẫn bị chấm FP + FN kép.
LABEL_ALIASES.update({
    "tire": {"wheel", "tyre"},
    "auto part": {"wheel", "tire", "bumper", "headlight", "windshield", "mirror"},
    "land vehicle": {"car", "truck", "bus", "van", "motorcycle", "bicycle", "vehicle"},
    "vehicle registration plate": {"license plate", "number plate", "plate"},
    "human head": {"head", "face", "human face"},
    "human body": {"body", "person", "torso"},
    "sports equipment": {"ball", "racket", "bat", "glove", "helmet", "ski"},
    "clothing": {"shirt", "jacket", "dress", "coat", "t-shirt", "pants", "trousers",
                 "trouser", "jeans", "uniform", "jersey", "shorts"},
    "footwear": {"shoe", "sneaker", "boot", "sandal", "cleat"},
})


def _hypernym_match(det_label, gt_label, max_up=6):
    # SỬA: 4 -> 6 bậc. Đo từ log Internet-50: GT "mammal" sót 5 lượt dù ảnh có horse/dog —
    # chuỗi horse -> equine -> odd-toed ungulate -> ungulate -> placental -> mammal cần 5 bậc.
    det_syns = wordnet.synsets(det_label.replace(" ", "_"), pos=wordnet.NOUN)
    gt_syns = set(wordnet.synsets(gt_label.replace(" ", "_"), pos=wordnet.NOUN))
    if not det_syns or not gt_syns:
        return False
    frontier = set(det_syns)
    for _ in range(max_up):
        nxt = set()
        for s in frontier:
            # CHỈ hypernyms thường, KHÔNG instance_hypernyms: nghĩa tên-riêng trong WordNet
            # (vd "Tree" = diễn viên Herbert Beerbohm Tree -> actor -> person) gây khớp nhầm
            # kiểu "tree" ~ "person".
            nxt.update(s.hypernyms())
        if nxt & gt_syns:
            return True
        frontier = nxt
        if not frontier:
            break
    return False


def labels_match(det_label, gt_label):
    d, g = det_label.lower().strip(), gt_label.lower().strip()
    if check_synonyms(d, g):
        return True
    if g in LABEL_ALIASES and d in LABEL_ALIASES[g]:
        return True
    if d in LABEL_ALIASES and g in LABEL_ALIASES[d]:
        return True
    return _hypernym_match(d, g)


def map_label_to_vocab(label, vocab):
    l = label.lower().strip()
    if l in vocab:
        return l
    for v in vocab:
        if labels_match(l, v):
            return v
    return None


def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


## Bước 4 — Suy luận Florence-2: `<OD>` + `<DENSE_REGION_CAPTION>` (+ grounding tag thủ công)

Ba nguồn detection, gộp lại rồi khử trùng lặp:
1. `<OD>` — nhãn ngắn, tin cậy cao nhất (pseudo-score 0.90).
2. `<DENSE_REGION_CAPTION>` — caption từng vùng → trích danh từ chính (pseudo-score 0.75). Bắt được vật
   `<OD>` bỏ qua, và cho từ vựng phong phú hơn.
3. `<CAPTION_TO_PHRASE_GROUNDING>` với `extra_tags` — bạn chỉ định từ muốn tìm (pseudo-score 0.85).


In [ ]:
FLORENCE_MAX_NEW_TOKENS = 1024
FLORENCE_NUM_BEAMS = 3
PSEUDO_SCORE = {"od": 0.90, "grounding": 0.85, "self": 0.80, "dense": 0.75, "base": 0.70}

# MỚI (phương pháp phủ từ vựng SẠCH hơn): SELF-GROUNDING — bảo Florence-2 tự mô tả ảnh thật chi
# tiết (<MORE_DETAILED_CAPTION>) rồi ground CHÍNH caption đó (<CAPTION_TO_PHRASE_GROUNDING>).
# Khác căn bản với base-vocab grounding: từ vựng đến từ những gì model THỰC SỰ THẤY trong ảnh
# ("a roasted turkey on a blue seat...") chứ không phải danh sách áp đặt từ ngoài -> gần như
# không ảo giác, không cần hàng rào phủ quyết. Nhãn trích qua caption_to_label nên vẫn bảo đảm
# từ điển + concrete filter.
ENABLE_SELF_GROUNDING = False

# ====== MỚI: từ vựng nền (giải quyết "RAM++ tốt hơn trên ảnh cá nhân") ======
# <OD> của Florence-2 chỉ đề xuất vật nó "chú ý" -> ảnh bừa bộn nhiều vật (bếp!) sẽ bỏ sót
# microwave/fridge/pot dù nhìn thấy rõ. Khi bật cờ này, pipeline ground THÊM một bộ từ vựng
# nền qua <CAPTION_TO_PHRASE_GROUNDING> (chia cụm ~12 từ/lần) -> phủ hết vật thông dụng
# giống cơ chế BASE_VOCAB bên bản RAM++/OWLv2.
# LƯU Ý: grounding có xu hướng "ảo giác" — từ KHÔNG có trong ảnh vẫn có thể được trả box.
# Đã chặn bớt bằng: (1) bỏ box trùm >85%% ảnh, (2) pseudo-score thấp (0.70) để thua khi dedup,
# (3) NÊN bật ENABLE_CLIP_SCORING = True ở Bước 2 khi dùng cờ này để CLIP gạn lần cuối.
ENABLE_BASE_VOCAB = False
# MỚI: chạy thêm <OD> trên 4 Ô CHỒNG LẤN (2x2, mỗi ô 60%% ảnh) — thuốc đặc trị vật NHỎ.
# Phân tích 126 ảnh COCO cho thấy nhóm sót nặng nhất là vật nhỏ: fork 67%%, mouse 100%%,
# scissors 100%%, cell phone 40%%... — quá nhỏ với một lần nhìn toàn ảnh của <OD>.
# Box chạm mép TRONG của ô (vật bị cắt cụt) sẽ bị bỏ để không sinh box cụt trùng lặp.
# Chi phí: +4 lần <OD>/ảnh (chậm hơn ~2x) — tắt nếu cần tốc độ.
ENABLE_TILED_OD = False
# MỚI: bộ từ vựng GIA DỤNG mở rộng (~50 từ) — tách riêng khỏi BASE_VOCAB để eval không chậm thêm;
# bật ENABLE_EXTRA_VOCAB = True ở cell ảnh cá nhân. An toàn nhờ CLIP biên độ gác cửa từng box.
ENABLE_EXTRA_VOCAB = False
# MỚI: grounding CÓ CHỦ ĐÍCH — chỉ ~12 từ sót nặng nhất đo được trên COCO (55/56 lượt sót là
# "sót thật": <OD>/dense không hề đề xuất box). Khác lần "FP nổ 1408": chỉ 1 chunk grounding
# thay vì 7, và mọi box vẫn qua đủ 3 lớp gác (chặn đè vật đã biết + sàn CLIP + biên độ).
ENABLE_TARGETED_GROUNDING = False
TARGETED_GROUNDING_WORDS = [
    "dining table", "bench", "couch", "fork", "teddy bear", "handbag",
    "tie", "traffic light", "mouse", "scissors", "cell phone", "bowl",
]
EXTRA_HOUSEHOLD_VOCAB = [
    "mug", "plate", "jar", "can", "kettle", "pan", "pot", "cutting board", "spatula",
    "ladle", "whisk", "tray", "jug", "pitcher", "glass", "napkin", "towel", "apron",
    "glove", "faucet", "stove", "cabinet", "drawer", "shelf", "basket", "bucket",
    "lamp", "candle", "mirror", "pillow", "blanket", "curtain", "rug", "hanger",
    "toy", "doll", "camera", "headphones", "speaker", "watch", "wallet", "key",
    "pen", "pencil", "notebook", "envelope", "monitor", "printer", "charger", "cable",
    # rau củ quả cụ thể — thay cho các từ phạm trù đã bị chặn ("food", "dinner"...)
    "kale", "lettuce", "tomato", "onion", "garlic", "pepper", "mushroom", "cucumber",
    "corn", "potato", "cabbage", "herb",
    # SỬA (bug turkey->sandwich): thịt/thực phẩm cụ thể. BASE_VOCAB (COCO) không có turkey/chicken
    # nên CLIP so crop con gà tây với các món CÓ SẴN (sandwich, pizza...) và "sandwich" thắng
    # một cách "hợp lệ". Đưa từ ĐÚNG vào cuộc đua thì từ sai thua biên độ ngay — nguyên tắc:
    # cửa CLIP chỉ công bằng khi danh sách ứng viên chứa đáp án đúng.
    "turkey", "chicken", "ham", "steak", "meat", "egg", "bread", "cheese", "sausage",
    "fish", "shrimp", "salad", "bell pepper", "watermelon", "grape", "strawberry", "lemon",
]
BASE_VOCAB = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "bench", "bird", "cat", "dog", "horse",
    "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag",
    "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
    "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
    "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
    "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant", "bed",
    "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
    "teddy bear", "hair drier", "toothbrush", "tablet",
]


@torch.no_grad()
def run_florence(image, task_prompt, text_input=None):
    prompt = task_prompt if text_input is None else task_prompt + text_input
    inputs = florence_processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    generated_ids = florence_model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=FLORENCE_MAX_NEW_TOKENS,
        num_beams=FLORENCE_NUM_BEAMS,
        do_sample=False,
    )
    text = florence_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return florence_processor.post_process_generation(
        text, task=task_prompt, image_size=(image.width, image.height),
    )[task_prompt]


# SỬA: các từ chỉ ĐỒ ĂN/CẢNH quá chung chung — đúng về ngữ nghĩa nhưng vô dụng làm flashcard
# ("food" không dạy được từ vựng gì; vật cụ thể bên dưới nó — carrot, bowl — mới đáng làm thẻ).
GENERIC_LABEL_BLACKLIST = {"food", "dinner", "lunch", "breakfast", "meal", "snack",
                           "dish", "cuisine", "produce", "object", "item", "thing", "stuff",
                           "groceries", "goods", "product"}


def _clean_label(raw_label, from_caption=False):
    """Chuẩn hoá nhãn thô của Florence-2 -> nhãn flashcard, None nếu bị loại."""
    if not raw_label:
        return None
    # SỬA: nhãn <OD> nhiều từ giờ CŨNG đi qua caption_to_label (trích cụm danh từ liên tiếp +
    # kiểm tra từ ghép WordNet). Bản cũ lấy thô "2 token cuối" nên "plug and socket" ra "and socket".
    label = caption_to_label(raw_label)
    if not label:
        return None
    if is_scene_background_word(label):
        return None
    if label in GENERIC_LABEL_BLACKLIST:
        return None   # quá chung chung, không có giá trị flashcard
    if ENABLE_CONCRETE_FILTER and is_concrete_object_word(label) is False:
        return None   # danh từ trừu tượng/động từ -> không phải vật chụp được
    head = label.split()[-1]
    # SỬA MỚI: nhãn trích từ CAPTION phải qua từ điển kể cả ở chế độ mềm — caption của Florence-2
    # đôi khi sinh TÊN RIÊNG ("kennedy", "heston", "john wayne") hoặc typo khi generate
    # ("taillilight") làm flashcard rác. Chế độ mềm chỉ dành cho nhãn <OD>/grounding.
    if from_caption and not is_valid_dictionary_word(head):
        return None
    if STRICT_DICTIONARY_FILTER and not is_valid_dictionary_word(head):
        return None
    return label


def florence_detect(image, extra_tags=None, use_dense_caption=True):
    """Trả về list[{"label", "score", "box", "source"}] — box pixel [x1,y1,x2,y2]."""
    detections = []

    # ---- 1. <OD> ----
    od = run_florence(image, "<OD>")
    for box, raw in zip(od.get("bboxes", []), od.get("labels", [])):
        label = _clean_label(raw, from_caption=False)
        if label:
            detections.append({"label": label, "score": PSEUDO_SCORE["od"],
                               "box": [float(v) for v in box], "source": "od"})

    # ---- 1b. <OD> trên 4 ô chồng lấn (xem chú thích ENABLE_TILED_OD) ----
    if ENABLE_TILED_OD:
        W, H = image.width, image.height
        tiles = [(0, 0, int(0.6 * W), int(0.6 * H)), (int(0.4 * W), 0, W, int(0.6 * H)),
                 (0, int(0.4 * H), int(0.6 * W), H), (int(0.4 * W), int(0.4 * H), W, H)]
        margin = 8
        for (tx1, ty1, tx2, ty2) in tiles:
            od_t = run_florence(image.crop((tx1, ty1, tx2, ty2)), "<OD>")
            for box, raw in zip(od_t.get("bboxes", []), od_t.get("labels", [])):
                x1, y1, x2, y2 = (float(v) for v in box)
                tw, th = tx2 - tx1, ty2 - ty1
                # bỏ box chạm mép TRONG của ô (vật bị ô cắt cụt) — mép trùng biên ảnh thì cho phép
                if ((x1 < margin and tx1 > 0) or (y1 < margin and ty1 > 0)
                        or (x2 > tw - margin and tx2 < W) or (y2 > th - margin and ty2 < H)):
                    continue
                label = _clean_label(raw, from_caption=False)
                if label:
                    detections.append({"label": label, "score": PSEUDO_SCORE["od"] - 0.02,
                                       "box": [x1 + tx1, y1 + ty1, x2 + tx1, y2 + ty1],
                                       "source": "od_tile"})

    # ---- 1c. Self-grounding: caption chi tiết -> ground chính caption đó ----
    if ENABLE_SELF_GROUNDING:
        try:
            detailed_cap = run_florence(image, "<MORE_DETAILED_CAPTION>")
            if isinstance(detailed_cap, str) and detailed_cap.strip():
                gr_self = run_florence(image, "<CAPTION_TO_PHRASE_GROUNDING>",
                                       text_input=detailed_cap.strip())
                for box, phrase in zip(gr_self.get("bboxes", []), gr_self.get("labels", [])):
                    label = _clean_label(phrase, from_caption=True)
                    if label:
                        detections.append({"label": label, "score": PSEUDO_SCORE["self"],
                                           "box": [float(v) for v in box], "source": "self"})
        except Exception as e:
            print(f"   (Self-grounding lỗi, bỏ qua: {e})")

    # ---- 2. <DENSE_REGION_CAPTION> ----
    if use_dense_caption:
        dense = run_florence(image, "<DENSE_REGION_CAPTION>")
        for box, cap in zip(dense.get("bboxes", []), dense.get("labels", [])):
            label = _clean_label(cap, from_caption=True)
            if label:
                detections.append({"label": label, "score": PSEUDO_SCORE["dense"],
                                   "box": [float(v) for v in box], "source": "dense"})

    # ---- 3. Grounding các tag thủ công (thay extra_tags cũ) ----
    if extra_tags:
        text = ". ".join(t.strip().lower() for t in extra_tags if t.strip()) + "."
        gr = run_florence(image, "<CAPTION_TO_PHRASE_GROUNDING>", text_input=text)
        for box, raw in zip(gr.get("bboxes", []), gr.get("labels", [])):
            label = (raw or "").strip().lower()
            if label:
                detections.append({"label": label, "score": PSEUDO_SCORE["grounding"],
                                   "box": [float(v) for v in box], "source": "grounding"})

    # ---- 4. Từ vựng nền (tuỳ chọn — xem chú thích ở đầu cell) ----
    if ENABLE_BASE_VOCAB or ENABLE_TARGETED_GROUNDING:
        img_area = float(image.width * image.height)
        existing = [d["label"] for d in detections]
        if ENABLE_BASE_VOCAB:
            query_vocab = list(BASE_VOCAB) + (EXTRA_HOUSEHOLD_VOCAB if ENABLE_EXTRA_VOCAB else [])
        else:
            query_vocab = list(TARGETED_GROUNDING_WORDS)
        todo = [w for w in query_vocab
                if not any(labels_match(w, e) or labels_match(e, w) for e in existing)]
        for k in range(0, len(todo), 12):
            chunk = todo[k:k + 12]
            text = ". ".join(chunk) + "."
            gr = run_florence(image, "<CAPTION_TO_PHRASE_GROUNDING>", text_input=text)
            for box, raw in zip(gr.get("bboxes", []), gr.get("labels", [])):
                label = (raw or "").strip().lower()
                if not label or label not in chunk:
                    continue   # chỉ nhận đúng từ trong chunk, bỏ cụm ghép lạ
                x1, y1, x2, y2 = (float(v) for v in box)
                if (x2 - x1) * (y2 - y1) > 0.85 * img_area:
                    continue   # box trùm gần hết ảnh = dấu hiệu ảo giác của grounding
                detections.append({"label": label, "score": PSEUDO_SCORE["base"],
                                   "box": [x1, y1, x2, y2], "source": "base"})

    return detections


def dedup_detections(detections, same_label_iou=0.5, related_label_iou=0.7):
    """NMS đơn giản 2 mức: cùng nhãn (IoU>=0.5) và nhãn "liên quan" theo labels_match (IoU>=0.7).
    Ưu tiên giữ nguồn điểm cao hơn (<OD> > grounding > dense)."""
    survivors = sorted(detections, key=lambda d: -d["score"])
    kept, suppressed = [], [False] * len(survivors)
    for i in range(len(survivors)):
        if suppressed[i]:
            continue
        kept.append(survivors[i])
        for j in range(i + 1, len(survivors)):
            if suppressed[j]:
                continue
            iou = compute_iou(survivors[i]["box"], survivors[j]["box"])
            same = survivors[i]["label"] == survivors[j]["label"]
            related = same or labels_match(survivors[j]["label"], survivors[i]["label"]) \
                           or labels_match(survivors[i]["label"], survivors[j]["label"])
            if (same and iou >= same_label_iou) or (related and iou >= related_label_iou):
                suppressed[j] = True
    return kept


## Bước 5 — Pipeline chính: Florence-2 → dedup → SAM xác thực + cắt flashcard

Khối SAM giữ nguyên bản tối ưu của v7/v8: multimask + stability score, dọn nhiễu morphology, loại box có
mask quá nhỏ (bộ lọc false-positive), crop theo bbox thực của mask + nền trong suốt (RGBA).


In [ ]:
import cv2
import os
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
from PIL import Image

SAM_IOU_WEIGHT = 0.7
SAM_STABILITY_WEIGHT = 0.3
SAM_MIN_MASK_AREA_PX = 400
SAM_MIN_MASK_BOX_RATIO = 0.05
SAM_CROP_PADDING = 4

EVAL_RESTRICT_VOCAB = None        # set(...) khi eval COCO: chỉ giữ detection ánh xạ được vào từ vựng
EVAL_RELAX_SMALL_FILTERS = False  # True khi eval COCO: nới ngưỡng mask nhỏ
MAX_CARDS_PER_LABEL = None        # chế độ flashcard đặt = 1: mỗi TỪ 1 thẻ (wheel x5 -> 1 thẻ).
                                  # Eval PHẢI để None vì COCO/OI chấm theo TỪNG instance.
REQUIRE_DICTIONARY_LABELS = True  # cổng cuối: mọi từ ra flashcard PHẢI có trong từ điển
CLIP_BASE_THRESHOLD = 0.23        # SỬA: nâng sàn 0.20 -> 0.23 CHỈ áp cho nguồn "base" — cốc góc trái
                                  # giờ do tiled OD bắt (nguồn od_tile, không qua cửa này), hết lý do
                                  # giữ sàn thấp; sàn thấp từng để lọt skis/envelope/key trong ảnh bếp.
# MỚI (nhắm word-precision 0.802 -> 0.85+): CLIP xác thực MỌI box (không chỉ nguồn base).
# Ngưỡng sàn riêng, mềm hơn base (od/dense là đề xuất thật của model, ít ảo giác hơn grounding).
ENABLE_CLIP_VERIFY_ALL = False
CLIP_VERIFY_ALL_THRESHOLD = 0.20
CLIP_RANK_MARGIN = 0.02           # nhãn của crop phải khớp KHÔNG KÉM QUÁ 0.02 so với từ khớp nhất
                                  # (fork/spoon/knife bám nhau ~0.01 vẫn sống; siết cửa cho ảo giác
                                  # "na ná": tennis racket từng thua handbag 0.043 mà vẫn lọt ở 0.03)


def _lazy_load_clip():
    """Tải CLIP khi cần (nếu chưa bật ENABLE_CLIP_SCORING ở Bước 2)."""
    global clip_model, clip_processor
    if "clip_model" in globals():
        return
    from transformers import CLIPModel, CLIPProcessor
    print("   (Tải CLIP ~600MB để xác thực từ vựng nền — chỉ lần đầu...)")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


@torch.no_grad()
def clip_verify_detections(raw_image, dets, threshold, margin=None):
    """SỬA v2 — kiểm tra theo THỨ HẠNG thay vì chỉ ngưỡng tuyệt đối. Ngưỡng tuyệt đối để lọt
    các ảo giác "sát ngưỡng" (frisbee/snowboard trong bếp đạt 0.22-0.24). Giờ: crop phải khớp
    với CHÍNH TỪ CỦA NÓ gần bằng từ khớp nhất — crop "frisbee" thực chất là cái bát sẽ khớp
    "bowl" mạnh hơn hẳn (chênh ~0.07) -> loại; fork/spoon/knife bám nhau ~0.01 -> sống.
    Điều kiện giữ: sim >= ngưỡng sàn VÀ sim >= (điểm cao nhất của crop - CLIP_RANK_MARGIN)."""
    if not dets:
        return [], []
    if margin is None:
        margin = CLIP_RANK_MARGIN
    _lazy_load_clip()
    crops = []
    for d in dets:
        x1, y1, x2, y2 = (int(v) for v in d["box"])
        crops.append(raw_image.crop((max(0, x1), max(0, y1), max(x1 + 1, x2), max(y1 + 1, y2))))

    # tập ứng viên = từ vựng nền + nhãn của chính các det (khử trùng, giữ thứ tự)
    cand_vocab = list(BASE_VOCAB) + (EXTRA_HOUSEHOLD_VOCAB if ENABLE_EXTRA_VOCAB else [])
    candidates = list(dict.fromkeys(cand_vocab + [d["label"] for d in dets]))
    text_inputs = clip_processor(text=[f"a photo of a {c}" for c in candidates],
                                 return_tensors="pt", padding=True).to(device)
    te = clip_model.get_text_features(**text_inputs)
    te = te / te.norm(dim=-1, keepdim=True)
    img_inputs = clip_processor(images=crops, return_tensors="pt").to(device)
    ie = clip_model.get_image_features(**img_inputs)
    ie = ie / ie.norm(dim=-1, keepdim=True)
    sims = ie @ te.T   # [số crop, số ứng viên]

    kept, dropped = [], []
    for i, d in enumerate(dets):
        own = candidates.index(d["label"])
        s = float(sims[i, own])
        best = float(sims[i].max())
        if s >= threshold and s >= best - margin:
            kept.append(d)
        else:
            best_lbl = candidates[int(sims[i].argmax().item())]
            dropped.append((d["label"], round(s, 3), f"thua '{best_lbl}' {round(best - s, 3)}"))
    return kept, dropped
CLIP_REJECT_THRESHOLD = 0.20


def _sam_stability_score(mask_logits, threshold=0.0, offset=1.0):
    inter = np.count_nonzero(mask_logits > (threshold + offset))
    union = np.count_nonzero(mask_logits > (threshold - offset))
    return float(inter) / float(union) if union > 0 else 0.0


def _sam_clean_mask(binary_mask):
    mask_u8 = (binary_mask.astype(np.uint8)) * 255
    kernel = np.ones((3, 3), np.uint8)
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel, iterations=1)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if n_labels <= 1:
        return binary_mask
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return labels == largest


def segment_one_box(box_np):
    x1, y1, x2, y2 = box_np
    point_coords = np.array([[(x1 + x2) / 2.0, (y1 + y2) / 2.0]])
    point_labels = np.array([1])
    raw_masks, iou_preds, _ = sam_predictor.predict(
        box=box_np, point_coords=point_coords, point_labels=point_labels,
        multimask_output=True, return_logits=True,
    )
    stability = np.array([_sam_stability_score(m) for m in raw_masks])
    best = int(np.argmax(SAM_IOU_WEIGHT * iou_preds + SAM_STABILITY_WEIGHT * stability))
    mask = raw_masks[best] > sam_predictor.model.mask_threshold
    return _sam_clean_mask(mask), float(iou_preds[best]), float(stability[best])


@torch.no_grad()
def clip_score_crops(crop_images, labels):
    inputs = clip_processor(text=[f"a photo of a {l}" for l in labels], images=crop_images,
                            return_tensors="pt", padding=True).to(device)
    out = clip_model(**inputs)
    img_e = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    txt_e = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return (img_e * txt_e).sum(dim=-1).cpu().tolist()


PIPELINE_VERSION = "F2-v13 (word-level + clip-verify-all, 2026-07-26)"   # NÂNG SỐ MỖI LẦN NHẬN FILE MỚI


def process_and_crop_flashcards(image_path, output_dir="flashcards", save_metadata=True,
                                 extra_tags=None, use_dense_caption=True, show_plot=True):
    print(f"[{PIPELINE_VERSION}]")   # dấu phiên bản — đối chiếu log để biết chắc đang chạy bản nào
    raw_image = Image.open(image_path).convert("RGB")
    image_np = np.array(raw_image)
    H, W = image_np.shape[:2]

    # ---------- 1. Florence-2: đề xuất box + nhãn mở ----------
    print("1. Florence-2 (<OD> + dense region caption)...")
    detections = florence_detect(raw_image, extra_tags=extra_tags, use_dense_caption=use_dense_caption)
    n_raw = len(detections)

    # ---------- 2. Giao từ vựng khi eval trên bộ nhãn đóng (COCO) ----------
    if EVAL_RESTRICT_VOCAB:
        mapped = []
        for d in detections:
            mv = map_label_to_vocab(d["label"], EVAL_RESTRICT_VOCAB)
            if mv:
                mapped.append({**d, "label": mv})
        print(f"-> [EVAL] Giao với từ vựng đánh giá: giữ {len(mapped)}/{len(detections)} detection.")
        detections = mapped

    # ---------- 3. Khử trùng lặp giữa <OD> / dense / grounding ----------
    detections = dedup_detections(detections)
    print(f"-> {n_raw} box thô -> {len(detections)} sau dedup: {[d['label'] for d in detections]}")

    # CLIP gác cửa cho từ vựng nền (bắt buộc khi ENABLE_BASE_VOCAB — xem clip_verify_detections)
    base_dets = [d for d in detections if d.get("source") == "base"]
    if base_dets:
        others = [d for d in detections if d.get("source") != "base"]
        # SỬA MỚI: box "base" đè lên vật MÀ <OD> ĐÃ NHẬN DIỆN với nhãn KHÁC -> tin <OD>, bỏ base.
        # Ảo giác grounding hay "mượn xác" vật thật: box "skis" đè lên thìa gỗ mà OD gọi là spoon —
        # biên độ CLIP không cứu nổi vì thìa gỗ dài trông giống skis thật.
        # LỊCH SỬ: từng có luật "phủ quyết" — bỏ box base đè lên vật nguồn khác (rồi thu hẹp còn
        # od/od_tile). Đo thật 2 vòng cho thấy luật này KHÔNG phân biệt nổi "đè vì ảo giác" với
        # "đè vì vật nằm TRÊN vật khác" (box turkey đè box toy = cái ghế con gà tây ngồi lên ->
        # giết oan turkey, mất 41 -> 24 thẻ). Nay CLIP biên độ đã có bộ ứng viên giàu (log cho
        # thấy skis/snowboard thua "ham"/"spatula" bằng biên độ) nên luật này THỪA -> tắt mặc
        # định, giữ code làm tư liệu ablation.
        ENABLE_BASE_OVERLAP_VETO = False
        if ENABLE_BASE_OVERLAP_VETO:
            _trusted = [o for o in others if o.get("source") in ("od", "od_tile")]

            def _overlaps_known(bd):
                for o in _trusted:
                    if compute_iou(bd["box"], o["box"]) > 0.6 and not (
                            labels_match(bd["label"], o["label"]) or labels_match(o["label"], bd["label"])):
                        return True
                return False
            n0 = len(base_dets)
            base_dets = [d for d in base_dets if not _overlaps_known(d)]
            if n0 - len(base_dets):
                print(f"-> [BASE] Bỏ {n0 - len(base_dets)} box từ vựng nền đè lên vật <OD> đã nhận diện.")
        kept, dropped = clip_verify_detections(raw_image, base_dets, CLIP_BASE_THRESHOLD)
        if dropped:
            print(f"-> [CLIP] Loại {len(dropped)} box từ vựng nền không khớp ảnh: {dropped}")
        detections = others + kept

    # CLIP xác thực toàn bộ (tuỳ chọn — ablation 4): loại box mà crop không khớp nhãn của chính nó
    if ENABLE_CLIP_VERIFY_ALL and detections:
        kept_all, dropped_all = clip_verify_detections(raw_image, detections, CLIP_VERIFY_ALL_THRESHOLD)
        if dropped_all:
            print(f"-> [CLIP-ALL] Loại {len(dropped_all)}/{len(detections)} box: {dropped_all[:10]}"
                  + (" ..." if len(dropped_all) > 10 else ""))
        detections = kept_all

    # Chế độ flashcard: mỗi label chỉ giữ box điểm cao nhất (xem chú thích MAX_CARDS_PER_LABEL)
    if MAX_CARDS_PER_LABEL:
        by_label = {}
        for d in sorted(detections, key=lambda x: -x["score"]):
            by_label.setdefault(d["label"], []).append(d)
        detections = [d for lst in by_label.values() for d in lst[:MAX_CARDS_PER_LABEL]]
        print(f"-> [FLASHCARD] Giữ tối đa {MAX_CARDS_PER_LABEL} thẻ/label: còn {len(detections)} thẻ.")

    # ---------- 4. SAM xác thực + cắt flashcard ----------
    print("2. SAM xác thực + cắt nền...")
    sam_predictor.set_image(image_np)
    os.makedirs(output_dir, exist_ok=True)

    if show_plot:
        fig, ax = plt.subplots(1, figsize=(10, 10))
        ax.imshow(raw_image)
    colors = cm.rainbow(np.linspace(0, 1, max(1, len(detections))))

    metadata, n_tiny_removed = [], 0
    min_mask_px = 50 if EVAL_RELAX_SMALL_FILTERS else SAM_MIN_MASK_AREA_PX

    for i, (d, color) in enumerate(zip(detections, colors)):
        # Cổng từ điển cuối: mọi từ lên flashcard phải có trong từ điển tiếng Anh
        if REQUIRE_DICTIONARY_LABELS and not is_valid_dictionary_word(d["label"].split()[-1]):
            continue
        x1, y1, x2, y2 = d["box"]
        mask, iou_pred, stab = segment_one_box(np.array(d["box"]))
        mask_area = int(mask.sum())
        box_area = max(1.0, (x2 - x1) * (y2 - y1))
        if mask_area < min_mask_px or (mask_area / box_area) < SAM_MIN_MASK_BOX_RATIO:
            n_tiny_removed += 1
            continue

        if show_plot:
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                           edgecolor=color, facecolor="none"))
            ax.text(x1, max(0, y1 - 5), f"{d['label']} [{d['source']}]", color=color,
                    fontsize=10, weight="bold")

        ys, xs = np.where(mask)
        mx1 = max(0, int(xs.min()) - SAM_CROP_PADDING); my1 = max(0, int(ys.min()) - SAM_CROP_PADDING)
        mx2 = min(W, int(xs.max()) + 1 + SAM_CROP_PADDING); my2 = min(H, int(ys.max()) + 1 + SAM_CROP_PADDING)
        if mx2 <= mx1 or my2 <= my1:
            n_tiny_removed += 1
            continue

        rgba = np.dstack([image_np, (mask.astype(np.uint8) * 255)])
        out_path = os.path.join(output_dir, f"{d['label'].replace(' ', '_')}_{i}.png")
        Image.fromarray(rgba[my1:my2, mx1:mx2], mode="RGBA").save(out_path)
        metadata.append({"label": d["label"], "score": d["score"], "box": d["box"],
                         "source": d["source"], "mask_area_px": mask_area,
                         "sam_iou_pred": round(iou_pred, 3), "sam_stability": round(stab, 3),
                         "file": out_path})

    if show_plot:
        ax.axis("off"); plt.show()

    # ---------- 5. CLIP chấm điểm thật (tuỳ chọn — Florence-2 không có confidence) ----------
    if ENABLE_CLIP_SCORING and metadata:
        print("3. CLIP chấm điểm crop <-> nhãn...")
        crops = [Image.open(m["file"]).convert("RGB") for m in metadata]
        sims = clip_score_crops(crops, [m["label"] for m in metadata])
        kept = []
        for m, s in zip(metadata, sims):
            if s >= CLIP_REJECT_THRESHOLD:
                kept.append({**m, "clip_score": round(s, 4), "score": round(0.5 * m["score"] + 0.5 * s, 4)})
            elif os.path.exists(m["file"]):
                os.remove(m["file"])
        print(f"-> CLIP giữ {len(kept)}/{len(metadata)}.")
        metadata = sorted(kept, key=lambda m: -m["score"])

    if save_metadata:
        with open(os.path.join(output_dir, "metadata.json"), "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

    final_detections = [{"label": m["label"], "score": m["score"], "box": m["box"]} for m in metadata]
    print(f"\n===== BÁO CÁO =====")
    print(f"Box thô Florence-2:            {n_raw}")
    print(f"Sau dedup:                     {len(detections)}")
    print(f"Bị loại do SAM (mask quá nhỏ): {n_tiny_removed}")
    print(f"Flashcard cuối cùng:           {len(final_detections)} -> thư mục '{output_dir}'")
    return final_detections, [d["label"] for d in detections], [d["label"] for d in detections]


## Bước 6 — Bộ đánh giá (bản đã sửa, giống notebook v8)

`labels_match` khi so nhãn, `allowed_labels` cho giao thức federated của Open Images, in cả macro lẫn
**micro-average** (con số nên dùng để so sánh giữa 2 notebook / các cấu hình).


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
from PIL import Image as _PILImage


def draw_eval_overlay(image_path, detections, ground_truth, metrics):
    """Vẽ 1 hình cho ảnh test — XANH LÁ nét đứt = ground truth, MÀU nét liền = dự đoán."""
    img = _PILImage.open(image_path).convert("RGB")
    fig, ax = plt.subplots(1, figsize=(11, 11))
    ax.imshow(img)
    for g in ground_truth:
        x1, y1, x2, y2 = g["box"]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                       edgecolor="lime", facecolor="none", linestyle="--"))
        tag = g["label"] + (" [GROUP]" if g.get("is_group") else "")
        ax.text(x1, min(img.height - 5, y2 + 14), f"GT: {tag}", color="lime",
                fontsize=8, weight="bold")
    colors = cm.rainbow(np.linspace(0, 1, max(1, len(detections))))
    for d, color in zip(detections, colors):
        x1, y1, x2, y2 = d["box"]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                       edgecolor=color, facecolor="none"))
        ax.text(x1, max(0, y1 - 5), f"{d['label']} ({d.get('score', 0):.2f})",
                color=color, fontsize=8, weight="bold")
    missed = ", ".join(metrics["missed_objects"][:8]) or "(không)"
    ax.set_title(f"P={metrics['precision']}  R={metrics['recall']}  F1={metrics['f1']}  "
                 f"(TP={metrics['true_positive']} FP={metrics['false_positive']} "
                 f"FN={metrics['false_negative']})\nBỏ sót: {missed}", fontsize=10)
    ax.axis("off")
    plt.show()


def evaluate_detection(detections, ground_truth, iou_threshold=0.5, allowed_labels=None):
    order = sorted(range(len(detections)), key=lambda i: -detections[i].get("score", 0.0))
    matched_gt, matched_det, ignored_det = set(), set(), set()

    def _ioa(det_box, gt_box):
        """Giao / diện tích DETECTION — chuẩn Open Images cho group box (IsGroupOf)."""
        x1 = max(det_box[0], gt_box[0]); y1 = max(det_box[1], gt_box[1])
        x2 = min(det_box[2], gt_box[2]); y2 = min(det_box[3], gt_box[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        det_area = max(1e-6, (det_box[2] - det_box[0]) * (det_box[3] - det_box[1]))
        return inter / det_area

    for di in order:
        d = detections[di]
        best_iou, best_gi, best_is_group = 0.0, -1, False
        for gi, g in enumerate(ground_truth):
            if not labels_match(d["label"], g["label"]):
                continue
            if g.get("is_group"):
                # group box: mọi box đơn so IoU đều trượt -> dùng IoA >= 0.5;
                # det đầu khớp = TP, các det sau cùng group = ignore
                ioa = _ioa(d["box"], g["box"])
                if ioa >= 0.5 and ioa > best_iou:
                    best_iou, best_gi, best_is_group = ioa, gi, True
            else:
                if gi in matched_gt:
                    continue
                iou = compute_iou(d["box"], g["box"])
                if iou > best_iou:
                    best_iou, best_gi, best_is_group = iou, gi, False
        if best_gi >= 0 and (best_is_group or best_iou >= iou_threshold):
            if best_is_group and best_gi in matched_gt:
                ignored_det.add(di)
            else:
                matched_gt.add(best_gi); matched_det.add(di)
        elif allowed_labels is not None and map_label_to_vocab(d["label"], allowed_labels) is None:
            ignored_det.add(di)   # Open Images federated: lớp chưa verify -> ignore, không phạt FP

    tp = len(matched_det)
    fp = len(detections) - tp - len(ignored_det)
    fn = len(ground_truth) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
        "true_positive": tp, "false_positive": fp, "false_negative": fn,
        "ignored_detections": len(ignored_det),
        "missed_objects": [g["label"] for gi, g in enumerate(ground_truth) if gi not in matched_gt],
        "extra_detections": [detections[di]["label"] for di in range(len(detections))
                             if di not in matched_det and di not in ignored_det],
    }


def evaluate_on_test_set(test_cases, extra_tags=None, label="", visualize=True):
    """visualize=True: vẽ overlay GT-vs-dự-đoán cho TỪNG ảnh test (đặt False nếu chỉ cần chỉ số)."""
    all_metrics = []
    tot_tp = tot_fp = tot_fn = 0
    w_tp = w_fp = w_fn = 0   # MỚI: đếm cấp độ TỪ (một từ dự đoán đúng nếu khớp bất kỳ từ GT nào,
                             # không xét IoU) — thước đo sát mục tiêu flashcard: "ảnh này pipeline
                             # gọi tên đúng những vật gì?", tách khỏi độ khít box.
    for case in test_cases:
        detections, _, _ = process_and_crop_flashcards(
            case["image_path"], save_metadata=False, show_plot=False,
            extra_tags=case.get("extra_tags", extra_tags),
        )
        m = evaluate_detection(detections, case["ground_truth"], allowed_labels=case.get("allowed_labels"))
        if visualize:
            draw_eval_overlay(case["image_path"], detections, case["ground_truth"], m)
        m["image"] = case["image_path"]
        all_metrics.append(m)
        tot_tp += m["true_positive"]; tot_fp += m["false_positive"]; tot_fn += m["false_negative"]
        print(f"[{case['image_path']}] P={m['precision']} R={m['recall']} F1={m['f1']}")
        if m["missed_objects"]:
            # In dạng text (trước đây chỉ nằm trong tiêu đề hình vẽ) để phân tích từ sót từ log
            print(f"   Bỏ sót: {sorted(set(m['missed_objects']))}")
        # cấp độ TỪ
        pred_words = {d["label"] for d in detections}
        gt_words = {g["label"] for g in case["ground_truth"]}
        allowed = case.get("allowed_labels")
        for p in pred_words:
            if any(labels_match(p, g) for g in gt_words):
                w_tp += 1
            elif allowed is None or map_label_to_vocab(p, allowed) is not None:
                w_fp += 1   # từ sai (nằm trong tập lớp đã verify); ngoài tập verify -> ignore
        w_fn += sum(1 for g in gt_words if not any(labels_match(p, g) for p in pred_words))

    avg_p = np.mean([m["precision"] for m in all_metrics])
    avg_r = np.mean([m["recall"] for m in all_metrics])
    avg_f1 = np.mean([m["f1"] for m in all_metrics])
    micro_p = tot_tp / (tot_tp + tot_fp) if (tot_tp + tot_fp) > 0 else 0.0
    micro_r = tot_tp / (tot_tp + tot_fn) if (tot_tp + tot_fn) > 0 else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0
    tag = f" ({label})" if label else ""
    print(f"\n===== KẾT QUẢ TRÊN {len(test_cases)} ẢNH TEST{tag} =====")
    print(f"MACRO (tb per-image):                P={avg_p:.3f} | R={avg_r:.3f} | F1={avg_f1:.3f}")
    print(f"MICRO (cộng dồn TP/FP/FN, NÊN DÙNG): P={micro_p:.3f} | R={micro_r:.3f} | F1={micro_f1:.3f} "
          f"(TP={tot_tp} FP={tot_fp} FN={tot_fn})")
    wp = w_tp / (w_tp + w_fp) if (w_tp + w_fp) else 0.0
    wr = w_tp / (w_tp + w_fn) if (w_tp + w_fn) else 0.0
    wf = 2 * wp * wr / (wp + wr) if (wp + wr) else 0.0
    print(f"WORD-LEVEL (độ phủ TỪ, không xét IoU — thước đo sát mục tiêu flashcard): "
          f"P={wp:.3f} | R={wr:.3f} | F1={wf:.3f}")
    return all_metrics


## Bước 7 — Bộ test A: COCO128

Đo với `EVAL_RESTRICT_VOCAB = 80 lớp COCO` (chỉ giữ detection ánh xạ được về 80 lớp — GT COCO không gán nhãn
gì khác nên detection ngoài 80 lớp dù đúng vẫn bị chấm FP) và `EVAL_RELAX_SMALL_FILTERS = True`.


In [ ]:
import glob
from PIL import Image

COCO80_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog",
    "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite",
    "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle",
    "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant",
    "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
    "teddy bear", "hair drier", "toothbrush",
]

COCO_SAMPLE_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
%cd /content
!wget -q -nc {COCO_SAMPLE_URL}
!unzip -oq coco128.zip -d /content/coco_sample_data


def load_labeled_test_cases(root_glob="/content/coco_sample_data/**/images/**/*.jpg"):
    test_cases = []
    for img_path in sorted(glob.glob(root_glob, recursive=True)):
        label_path = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        if not os.path.exists(label_path):
            continue
        w, h = Image.open(img_path).size
        ground_truth = []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = (float(v) for v in parts[1:])
                xc, yc, bw, bh = xc * w, yc * h, bw * w, bh * h
                lb = COCO80_CLASSES[cls_id] if cls_id < len(COCO80_CLASSES) else str(cls_id)
                ground_truth.append({"label": lb, "box": [xc - bw/2, yc - bh/2, xc + bw/2, yc + bh/2]})
        if ground_truth:
            test_cases.append({"image_path": img_path, "ground_truth": ground_truth})
    return test_cases


coco128_test_cases = load_labeled_test_cases()
print(f"Đã tải {len(coco128_test_cases)} ảnh COCO128 có nhãn thật.")


In [ ]:
# Chế độ đo công bằng trên bộ nhãn đóng 80 lớp
STRICT_DICTIONARY_FILTER = True
EVAL_RESTRICT_VOCAB = set(COCO80_CLASSES)
EVAL_RELAX_SMALL_FILTERS = True
MAX_CARDS_PER_LABEL = None

# KẾT LUẬN THỰC NGHIỆM: TẮT máy phủ từ vựng khi đo benchmark. Lần chạy có bật: recall tăng đúng
# thiết kế (0.666 -> 0.740) nhưng FP nổ 383 -> 1408, F1 sập 0.641 -> 0.454. Nguyên nhân CHÍNH
# (thấy rõ trong log CLIP): <CAPTION_TO_PHRASE_GROUNDING> trả box cho gần như MỌI từ được hỏi —
# 80 từ COCO/ảnh = hàng chục box ảo giác/ảnh, CLIP chặn phần lớn nhưng phần lọt vẫn nhấn chìm
# precision; tiled OD góp thêm box trùng/cụt. => Cấu hình BENCHMARK (ở đây, cờ TẮT, F1 0.641 —
# số dùng cho báo cáo) tách khỏi cấu hình SẢN PHẨM (ảnh cá nhân, cờ BẬT, tối ưu độ phủ TỪ).
# Báo cáo nên trình bày cả hai cấu hình và lý do tách.
ENABLE_BASE_VOCAB = False
ENABLE_TILED_OD = False
ENABLE_TARGETED_GROUNDING = False   # SỬA: reset ĐỦ 4 cờ phủ vocab — nếu vừa chạy ảnh cá nhân
ENABLE_SELF_GROUNDING = False       # hoặc dừng ablation giữa chừng, cờ dính True sẽ làm bẩn
                                    # số benchmark một cách âm thầm
ENABLE_CLIP_VERIFY_ALL = False

coco128_metrics = evaluate_on_test_set(coco128_test_cases, label="COCO128 / Florence-2")


## Bước 7b (TÙY CHỌN) — Ablation nâng recall benchmark

Phân tích từ sót COCO cho thấy **55/56 lượt sót là "sót thật"** — không có cả nhãn tổng quát hơn trong
ảnh, nghĩa là alias/matching không cứu được; phải tăng nguồn đề xuất box. Hai đòn bẩy, đo **tách biệt**
để biết mỗi cái trả giá bao nhiêu precision (bài học từ lần FP nổ 383 → 1408):

1. **Tiled OD** — nhắm vật nhỏ (fork sót 67%, mouse/scissors 100%, cell phone, tie, traffic light).
2. **Targeted grounding** — grounding chỉ ~12 từ hay sót (dining table 70%, bench 60%, couch...),
   không phải 80 từ, và đi qua đủ 3 lớp gác.

So MICRO F1 từng ablation với mốc **0.642**; cấu hình nào thắng thì nâng thành benchmark chính thức.
Mỗi ablation chạy lại đủ 126 ảnh — tổng ~2 lần thời gian eval thường.


In [ ]:
# --- Ablation 1: chỉ tiled OD ---
ENABLE_TILED_OD = True
ENABLE_BASE_VOCAB = False
ENABLE_TARGETED_GROUNDING = False
coco_tiled = evaluate_on_test_set(coco128_test_cases, label="COCO128 / +tiled OD", visualize=False)

# --- Ablation 2: chỉ targeted grounding ---
ENABLE_TILED_OD = False
ENABLE_TARGETED_GROUNDING = True
coco_targeted = evaluate_on_test_set(coco128_test_cases, label="COCO128 / +targeted grounding", visualize=False)

# --- Ablation 3: self-grounding (caption chi tiết -> ground chính nó) ---
ENABLE_TILED_OD = False
ENABLE_TARGETED_GROUNDING = False
ENABLE_SELF_GROUNDING = True
coco_selfg = evaluate_on_test_set(coco128_test_cases, label="COCO128 / +self-grounding", visualize=False)

# --- Ablation 4: CLIP xác thực MỌI box (nhắm word-precision) ---
ENABLE_SELF_GROUNDING = False
ENABLE_CLIP_VERIFY_ALL = True
coco_clipall = evaluate_on_test_set(coco128_test_cases, label="COCO128 / +CLIP verify all", visualize=False)

# --- Trả cờ về cấu hình benchmark gốc ---
ENABLE_TILED_OD = False
ENABLE_TARGETED_GROUNDING = False
ENABLE_SELF_GROUNDING = False
ENABLE_CLIP_VERIFY_ALL = False


## Bước 8 — Bộ test B: Internet-50 (Open Images V7, giao thức federated)

Không giao vocab (mục tiêu là đo khả năng gọi tên từ mới); FP oan xử lý bằng `allowed_labels`
(= lớp có GT box ∪ positive/negative labels đã verify của từng ảnh).

> Nếu `import fiftyone` lỗi: chạy `!pip install -q fiftyone` ở cell mới rồi **Restart session** và chạy lại
> từ Bước 2 (fiftyone nâng cấp Pillow giữa phiên gây lỗi import PIL).


In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# SỬA: nếu phiên trước đã tạo dataset cùng tên, FiftyOne TÁI DÙNG bản cũ và bỏ qua label_types mới
if fo.dataset_exists("internet50_openimages_florence"):
    fo.delete_dataset("internet50_openimages_florence")

oi_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections", "classifications"],
    max_samples=50,
    shuffle=True,
    seed=51,
    dataset_name="internet50_openimages_florence",
)
print(f"Đã tải {len(oi_dataset)} ảnh Open Images V7 kèm ground truth.")


def load_openimages_test_cases(dataset):
    from PIL import Image as _PILImage
    test_cases = []
    for sample in dataset:
        try:
            img_w, img_h = _PILImage.open(sample.filepath).size
        except Exception as e:
            print(f"Bỏ qua {sample.filepath}: {e}")
            continue
        # SỬA: tải MỘT label type -> field tên "ground_truth"; tải NHIỀU loại (thêm "classifications")
        # -> mỗi loại mang tên riêng: "detections", "positive_labels", "negative_labels". Thử cả hai.
        gt_field = None
        for fname in ("detections", "ground_truth"):
            try:
                gt_field = sample[fname]
                if gt_field is not None:
                    break
            except (AttributeError, KeyError):
                continue
        dets = gt_field.detections if (gt_field is not None and getattr(gt_field, "detections", None)) else []
        ground_truth = []
        for det in dets:
            label = det.label.replace("_", " ").strip().lower()
            x, y, w, h = det.bounding_box
            # group box (IsGroupOf): 1 box trùm cả đám vật cùng loại -> eval xử lý bằng IoA
            is_group = False
            try:
                is_group = bool(det["IsGroupOf"])
            except Exception:
                pass
            ground_truth.append({"label": label, "is_group": is_group,
                                 "box": [x * img_w, y * img_h, (x + w) * img_w, (y + h) * img_h]})
        allowed_labels = {g["label"] for g in ground_truth}
        for field in ("positive_labels", "negative_labels"):
            cls_field = getattr(sample, field, None)
            if cls_field is not None and getattr(cls_field, "classifications", None):
                for c in cls_field.classifications:
                    allowed_labels.add(c.label.replace("_", " ").strip().lower())
        if ground_truth:
            test_cases.append({"image_path": sample.filepath, "ground_truth": ground_truth,
                               "allowed_labels": allowed_labels})
    return test_cases


internet50_test_cases = load_openimages_test_cases(oi_dataset)
print(f"Đã chuyển {len(internet50_test_cases)}/{len(oi_dataset)} ảnh sang test case.")


In [ ]:
# Chế độ open-vocab thật: bộ lọc mềm, không giao vocab, giữ lọc vật nhỏ như sản phẩm
STRICT_DICTIONARY_FILTER = False
EVAL_RESTRICT_VOCAB = None
EVAL_RELAX_SMALL_FILTERS = False
MAX_CARDS_PER_LABEL = None
ENABLE_BASE_VOCAB = False   # xem kết luận thực nghiệm ở cell eval COCO
ENABLE_TILED_OD = False
ENABLE_TARGETED_GROUNDING = False   # reset đủ cờ (lý do: xem cell eval COCO)
ENABLE_SELF_GROUNDING = False
ENABLE_CLIP_VERIFY_ALL = False

internet50_metrics = evaluate_on_test_set(internet50_test_cases, label="Internet-50 / Florence-2")


## Bước 8b — Vẽ lại khoanh vùng cho MỘT ảnh bất kỳ

`evaluate_on_test_set` (visualize=True) đã vẽ overlay cho từng ảnh khi chạy eval. Cell này dùng khi muốn
soi lại **một ảnh cụ thể** mà không chạy lại cả bộ: `visualize_test_case(internet50_test_cases[i])`
(hoặc `coco128_test_cases[i]`).


In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
from PIL import Image as _PILImage


def visualize_test_case(case, iou_threshold=0.5):
    """Vẽ 1 ảnh test: GT (xanh lá, nét đứt) vs dự đoán (màu, nét liền) + P/R/F1 của ảnh đó."""
    detections, _, _ = process_and_crop_flashcards(
        case["image_path"], save_metadata=False, show_plot=False,
        extra_tags=case.get("extra_tags"),
    )
    m = evaluate_detection(detections, case["ground_truth"],
                           iou_threshold=iou_threshold, allowed_labels=case.get("allowed_labels"))

    img = _PILImage.open(case["image_path"]).convert("RGB")
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.imshow(img)

    for g in case["ground_truth"]:
        x1, y1, x2, y2 = g["box"]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                       edgecolor="lime", facecolor="none", linestyle="--"))
        tag = g["label"] + (" [GROUP]" if g.get("is_group") else "")
        ax.text(x1, min(img.height - 5, y2 + 14), f"GT: {tag}", color="lime", fontsize=9, weight="bold")

    colors = cm.rainbow(np.linspace(0, 1, max(1, len(detections))))
    for d, color in zip(detections, colors):
        x1, y1, x2, y2 = d["box"]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                       edgecolor=color, facecolor="none"))
        ax.text(x1, max(0, y1 - 5), f"{d['label']} ({d.get('score', 0):.2f})",
                color=color, fontsize=9, weight="bold")

    ax.set_title(f"P={m['precision']}  R={m['recall']}  F1={m['f1']}  "
                 f"(TP={m['true_positive']} FP={m['false_positive']} FN={m['false_negative']})\n"
                 f"Bỏ sót: {m['missed_objects']}", fontsize=10)
    ax.axis("off")
    plt.show()
    return m


# Vẽ 3 ảnh ngẫu nhiên trong bộ Internet-50 (đổi sang coco128_test_cases nếu muốn xem COCO)
random.seed(7)
for case in random.sample(internet50_test_cases, k=min(3, len(internet50_test_cases))):
    visualize_test_case(case)


## Bước 9 — Tổng hợp & chạy ảnh riêng

So sánh trực tiếp với notebook v8 (RAM++ → detector): chạy cùng 2 bộ test, so cột **MICRO F1**. Điểm mạnh
kỳ vọng của Florence-2: ít khớp nối rò rỉ lỗi, nhãn dense-caption phong phú (tốt cho học từ mới); điểm yếu:
không có confidence thật (bật `ENABLE_CLIP_SCORING` nếu cần lọc), và có xu hướng ra ÍT box hơn trên ảnh
đông vật thể (recall vật nhỏ có thể thấp hơn detector chuyên dụng).


In [ ]:
import numpy as np


def summarize(metrics_list):
    tp = sum(m["true_positive"] for m in metrics_list)
    fp = sum(m["false_positive"] for m in metrics_list)
    fn = sum(m["false_negative"] for m in metrics_list)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(f1, 3), "n_images": len(metrics_list)}


coco_summary = summarize(coco128_metrics)
internet_summary = summarize(internet50_metrics)

print(f"{'Bộ test (micro)':<30}{'Ảnh':>6}{'Precision':>12}{'Recall':>10}{'F1':>8}")
print("-" * 66)
print(f"{'COCO128':<30}{coco_summary['n_images']:>6}{coco_summary['precision']:>12}"
      f"{coco_summary['recall']:>10}{coco_summary['f1']:>8}")
print(f"{'Internet-50 (Open Images)':<30}{internet_summary['n_images']:>6}{internet_summary['precision']:>12}"
      f"{internet_summary['recall']:>10}{internet_summary['f1']:>8}")


In [ ]:
# ---- Chạy thử ảnh của riêng bạn (chế độ sản phẩm) ----
STRICT_DICTIONARY_FILTER = False
ENABLE_BASE_VOCAB = True
MAX_CARDS_PER_LABEL = 1    # flashcard: mỗi từ 1 thẻ
ENABLE_TILED_OD = True     # bắt vật nhỏ (thìa, điện thoại, đồng hồ...)
ENABLE_SELF_GROUNDING = True  # phủ từ vựng bằng chính caption của model — sạch hơn base-vocab
ENABLE_TARGETED_GROUNDING = False  # khai báo tường minh đủ cờ để không dính trạng thái cell khác
ENABLE_CLIP_VERIFY_ALL = False     # ảnh cá nhân: CLIP đã gác nguồn base; bật thử nếu muốn siết thêm
ENABLE_EXTRA_VOCAB = True  # thêm ~50 từ gia dụng (mug, plate, kettle...) — CLIP gác từng box   # phủ vật thông dụng cho ảnh thật (xem chú thích ở Bước 4);
                           # khuyến nghị bật kèm ENABLE_CLIP_SCORING=True ở Bước 2 để gạn ảo giác
EVAL_RESTRICT_VOCAB = None
EVAL_RELAX_SMALL_FILTERS = False

from google.colab import files

print("Vui lòng tải lên 1 tấm ảnh để test:")
uploaded = files.upload()
test_image_path = list(uploaded.keys())[0]

# extra_tags: từ bạn muốn Florence-2 tìm thêm (qua <CAPTION_TO_PHRASE_GROUNDING>)
detections, labels, _ = process_and_crop_flashcards(test_image_path, extra_tags=None)
print("Nhãn flashcard:", [d["label"] for d in detections])
